# AAPL — bronze bars → silver `fact_bars_1m`

One row per minute of every regular session (09:30–15:59 ET: 390 rows on a full day,
210 on an early close), built as the cross product of the market calendar and the
minutes of each session, with the bronze bars left-joined onto it.

**Gap policy.** A minute with no trades carries the previous close forward
(`open = high = low = vwap = close`, `volume = 0`, `trade_count = 0`, `is_imputed = true`).
Prices are never interpolated linearly — that invents movement that did not happen and
contaminates every volatility estimate downstream. `is_imputed` is kept because it is
itself a feature (it marks illiquidity); it is the *target* that must exclude those rows.

**Adjustment.** Applied here, not at extraction time, from the versioned corporate
actions table — so the same code over the same bronze data always yields the same series.

## Utils

### Libraries & paths

In [ ]:
import logging
import os
import pathlib
import sys

from dotenv import load_dotenv


def _find_data_etl_root() -> pathlib.Path:
    """Locate the data-etl root, whatever the kernel decided the working directory is.

    JupyterLab sets it to the notebook directory, VS Code to the workspace root (local
    kernel) or to the Jupyter server root (kernel inside the container), so walking up
    from `cwd` alone is not enough.
    """
    candidates = [
        pathlib.Path(candidate)
        for candidate in (os.getenv("DATA_ETL_ROOT"), globals().get("__vsc_ipynb_file__"))
        if candidate
    ]
    candidates.append(pathlib.Path.cwd())

    for candidate in candidates:
        candidate = candidate.resolve()
        for parent in (candidate, *candidate.parents):
            if (parent / "src" / "extractions").is_dir():
                return parent

    for pattern in ("*/src/extractions", "*/*/src/extractions"):
        for match in sorted(pathlib.Path.cwd().glob(pattern)):
            return match.parents[1]

    raise RuntimeError("data-etl root not found; set the DATA_ETL_ROOT environment variable")


DATA_ETL_ROOT = _find_data_etl_root()
sys.path.insert(0, str(DATA_ETL_ROOT))

load_dotenv(DATA_ETL_ROOT.parent / ".env")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

import pandas as pd

from src.dwh.silver import bars_1m
from src.storage.catalog import get_catalog

### Parameters

In [ ]:
SYMBOL = "AAPL"
START_DATE = None                    # None = the whole range present in bronze
END_DATE = None

# Reads lakehouse.bronze.*, writes lakehouse.silver.fact_bars_1m.
lakehouse = get_catalog()
print(f"catalog {lakehouse.catalog} at {lakehouse.client.uri}, tables under {lakehouse.warehouse_root}")

## Build

`is_validated=True` is the contract: any violation raises and nothing is written.
The mandatory checks are

1. `low <= min(open, close) <= max(open, close) <= high`
2. `vwap` inside `[low, high]`
3. `volume > 0` if and only if `trade_count > 0`
4. 390 rows per full session, 210 per early close
5. no duplicates on `(symbol, timestamp_at)`
6. no 1-minute return above 20% that a corporate action does not explain

In [ ]:
silver = bars_1m.run_bronze_to_silver(
    SYMBOL,
    start=START_DATE,
    end=END_DATE,
    lakehouse=lakehouse,
    is_validated=True,
)

print(f"{len(silver):,} rows, {silver['session_date'].min()} \u2192 {silver['session_date'].max()}")
silver.head()

## Checks

In [ ]:
# Every session must be complete: 390 minutes, or 210 on an early close.
rows_per_session = silver.groupby("session_date").size()
rows_per_session.value_counts()

In [ ]:
# Imputed minutes should stay well under 1% in regular hours; above that, review the feed.
imputed_share = silver["is_imputed"].mean()
print(f"imputed minutes: {imputed_share:.3%}")

In [ ]:
# The 4:1 split of 2020-08-31: raw prices jump, adjusted prices do not.
around_split = (
    silver[silver["session_date"].between(pd.Timestamp("2020-08-27").date(),
                                          pd.Timestamp("2020-09-02").date())]
    .groupby("session_date")[["close", "close_adj", "adj_factor", "volume", "volume_adj"]]
    .last()
)
around_split

In [ ]:
# Distribution sanity: 1-minute log returns should have mean ~0 and fat tails (kurtosis > 3).
# A gaussian-looking distribution here means there is a bug.
import numpy as np

ordered = silver.sort_values(["session_date", "minute_index"])
ret_1m = np.log(ordered["close_adj"] / ordered["close_adj"].shift(1)).where(
    ordered["session_date"] == ordered["session_date"].shift(1)
)

print(f"mean {ret_1m.mean():.2e} | std {ret_1m.std():.2e} | kurtosis {ret_1m.kurtosis():.1f}")